## Premiere Cellule "nettoyage de la Database"

In [29]:
import pandas as pd

# Chargement du csv
chemin_fichier = '/home/greg/TARDIS/project_dataset.csv'
df = pd.read_csv(chemin_fichier, sep=';')

# On netoies tous les nombres (remplacer les virgules par des points)
colonnes_nombres = [
    "Retard moyen de tous les trains à l'arrivée",
    "Durée moyenne du trajet",
    "Retard moyen de tous les trains au départ",
    "Retard moyen des trains en retard au départ",
    "Retard moyen des trains en retard à l'arrivée"
]

for col in colonnes_nombres:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')

# On uniformisation les Dates (Le CSV mélange '2018-01' et '2024/03')
df['Date'] = df['Date'].astype(str).str.replace('/', '-')

#  On nettoie des colonnes inutiles
colonnes_inutiles = [c for c in df.columns if 'Commentaire' in c]
df = df.drop(columns=colonnes_inutiles)

# Si y'a pas la durée du trajet ou le retard d'arrivée, la ligne est bz elle sert à r
df = df.dropna(subset=["Retard moyen de tous les trains à l'arrivée", "Durée moyenne du trajet"])

# On fill les trous
df = df.fillna(0)

# nouvelle database
chemin_clean = '/home/greg/TARDIS/cleaned_dataset.csv'
df.to_csv(chemin_clean, index=False, sep=';')

print(f"Le fichier propre a été sauvegardé ici : {chemin_clean}")
print(f"Nombre de lignes prêtes pour l'IA : {len(df)}")

Le fichier propre a été sauvegardé ici : /home/greg/TARDIS/cleaned_dataset.csv
Nombre de lignes prêtes pour l'IA : 11476


## Cellule RandomForestRegressor

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import joblib

df = pd.read_csv('/home/greg/TARDIS/cleaned_dataset.csv', sep=';')
df['Date'] = df['Date'].astype(str).str.replace('/', '-').str.replace(' ', '-')
df['Date'] = pd.to_datetime(df['Date'], format='mixed', errors='coerce')
df = df.dropna(subset=['Date'])
df['Mois'] = df['Date'].dt.month

le_dep = LabelEncoder()
le_arr = LabelEncoder()

df['Gare_depart_enc'] = le_dep.fit_transform(df['Gare de départ'])
df['Gare_arrivee_enc'] = le_arr.fit_transform(df["Gare d'arrivée"])

X = df[['Gare_depart_enc', 'Gare_arrivee_enc', 'Mois', 'Durée moyenne du trajet']]
y = df["Retard moyen de tous les trains à l'arrivée"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Entraînement du modèle en cours...")

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
print(f"Entraînement terminé: {score:.3f}")

joblib.dump(model, '/home/greg/TARDIS/model.joblib')
joblib.dump(le_dep, '/home/greg/TARDIS/le_dep.joblib')
joblib.dump(le_arr, '/home/greg/TARDIS/le_arr.joblib')

print("Modèle et encodeurs sauvegardés dans le dossier TARDIS")


Entraînement du modèle en cours...
Entraînement terminé: 0.054
Modèle et encodeurs sauvegardés dans le dossier TARDIS
